# Chatbot and RAG Evaluation

  Retrival Augmented Generation(RAG) is a technique that enhances the Large Language Model(LLMs) by providing them with relavent external knowledge. It becomes one of the most widely used approaches for building LLM applications

 How to evaluate the RAG applications using LangSmith
 1.How to create test datasets
 2.How to run your RAG applications on those datasets\n
 3.How to measure your application's performance using different evaluation metrics\n


### Overview

A typical RAG evaluation workflow consists of three main steps:

1.Creating a dataset with questions and there expected answer
2.Running your RAG applications on those questions
3.Using evaluators to measure how well your pplicaton performed, looking at factors like:

Answer relevance
Answer accuracy'
Retrieval quality

We will create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog post


### ChatBot Evaluation

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGSMITH_API_KEY']=os.getenv('LANGSMITH_API_KEY')
os.environ['OPENROUTER_API_KEY']=os.getenv('OPENROUTER_API_KEY')
os.environ['LANGSMITH_TRACING']="true"



In [ ]:
## create the datapoints

from langsmith import Client
client=Client()

dataset_name="Simple Evaluation Chatbot"
dataset=client.create_dataset(dataset_name=dataset_name)


client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "LangChain is a framework for building LLM applications"}
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "LangSmith is a platform for building and evaluating LLM applications"}
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "OpenAI is a company that builds AI models"}
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "Google is a company that builds search engines"}
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "Mistral is a company that builds AI models"}
        },
    ]

)



{'example_ids': ['9f4e397c-7fe9-41f8-a9bb-783bc9bbd1d2',
  'a5d357dc-81ae-45ec-a4ca-eeb233dfd02b',
  '1af0af43-4dca-420c-a627-628d2a343f5c',
  '3387ca69-7fcd-4cab-a8b1-bfa543e8e86f',
  '53b3b3ee-2f5d-4a8c-a698-d324cf437330'],
 'count': 5,
 'as_of': '2026-10-10T08:57:05.289322395Z'}

### Define the Metrics(LLM as a judge)

In [19]:
import openai
from langsmith import wrappers

openai_client=wrappers.wrap_openai(openai.OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
))

eval_instructions="""you are an expert profeesor specialized in grading students answeto the questions"""

def corectness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content=f"""
    you are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT OR INCORRECT
    Grade:
    
    """
    response=openai_client.chat.completions.create(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        temperature=0.0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ],
    )
    return response.choices[0].message.content.strip() == "CORRECT"

In [20]:
## Concision - check whether the actual output is less than 2x the length of the expected result

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs['response'])) < 2 * int(len(reference_outputs['answer']))

### Run Evaluation

In [21]:
default_instructions="Respond to the users question in a short and concise manner (one short sentence)"
def my_app(question: str, model: str = "nvidia/nemotron-3-super-120b-a12b:free", instructions: str=default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question}
        ],
    ).choices[0].message.content



In [22]:
# Call my_app for every datapoints
def ls_target(inputs: str)->str:
    return {"response": my_app(inputs['question'])}

In [23]:
## Run our evaluation
experimental_results=client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[corectness, concision],
    experiment_prefix="Evaluation Chatbot",
)

View the evaluation results for experiment: 'Evaluation Chatbot-e6ac2b9a' at:
https://smith.langchain.com/o/10871889-9996-4579-886b-05660b35518f/datasets/4e350f0f-4d75-4850-940d-53d4c3fab4cc/compare?selectedSessions=2a82607b-7b78-4552-b6dc-5f9d1d1d008a




5it [00:33,  6.74s/it]
